# Phase 1 - Exploration

Before writing any pipeline code, I wanted to answer two things from one month of data (Jan 2025):

- **Objective 1** — what's the granularity and which columns actually matter?
- **Objective 2** — how big is one month, and what does that mean for storing 85 of them on a free hosting tier?

## Objective 1 — Granularity & Columns

### 1a) Data Granularity

- One row per flight. Every scheduled commercial flight (operated or cancelled) is its own row.
- Both on-time and delayed flights are included — both are necessary to compute delay *rates* (e.g. delay rate per carrier = delayed / total).

In [ ]:
import requests, zipfile, urllib3
import pandas as pd
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# - Pull Jan 2025 zip into the raw dir
# - Read CSV straight out of the zip, no temp extract
RAW_DIR = Path("raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

URL = ("https://transtats.bts.gov/PREZIP/"
       "On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_1.zip")
zip_path = RAW_DIR / "2025_01.zip"

if not zip_path.exists():
    r = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"},
                     stream=True, verify=False, timeout=300)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    print(f"Downloaded {zip_path.stat().st_size / 1e6:.1f} MB")

with zipfile.ZipFile(zip_path) as z:
    inner = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    with z.open(inner) as f:
        df = pd.read_csv(f, low_memory=False)

df.head()

### 1b) Columns

**Must-have (directly answer the prompt):**
- `FlightDate` — date of the flight. Drives the time range slider and the week/month/quarter/year toggle.
- `ArrDel15`, `DepDel15` — 0/1 flag for arrival/departure delayed > 15 min. This is the prompt's delay definition so it's the core filter.
- `Marketing_Airline_Network` — airline on the ticket (e.g. `DL`), 10 codes. This is the "Carrier" breakdown.
- `Origin`, `OriginCityName` — origin airport (IATA) and city.
- `Dest`, `DestCityName` — destination airport (IATA) and city.
- `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` — minutes of delay by cause (wide). The "Cause" breakdown. Probably worth melting to long when building the agg.
- `Cancelled`, `CancellationCode` — 0/1 cancel flag + reason (A/B/C/D = carrier/weather/NAS/security). Cancellations use the same cause taxonomy as delays.

**Helpful (likely call questions):**
- `OriginState`, `DestState` — lets the geography chart roll city up to state.
- `DayOfWeek` — 1-7. "Are Fridays the worst?"
- `DepTimeBlk` — hour block (e.g. `0600-0659`). "Do delays compound through the day?"
- `Distance` — miles. Short-haul vs long-haul.
- `Operating_Airline` — who actually flew it (21 codes, includes regionals like SkyWest). Useful if asked whether regionals are worse than mainlines.
- `Tail_Number` — unique plane ID (e.g. `N12345`). For a "worst planes" ranking.
- `ArrDelayMinutes` — delay magnitude so charts can show avg minutes late, not just % rate.
- `Diverted` — 0/1. Third disruption outcome alongside delay and cancellation.

In [ ]:
# Scan every column for:
# - dtype, null %, unique count, up to 10 sample values
# Drives the keep/drop decisions below.
rows = []
for c in df.columns:
    s = df[c]
    uniq = s.dropna().unique()
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "null_pct": round(100 * s.isna().sum() / len(s), 1),
        "n_unique": len(uniq),
        "samples": list(uniq[:10]),
    })

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

## Objective 2 — Data Size

**Raw:**
- One month (Jan 2025) = ~599k rows × 120 columns, uncompressed CSV ~300 MB.
- 85 months (Jan 2018 → Jan 2025) ≈ ~25 GB uncompressed CSV.

**After column pruning (~20 of 120 cols kept):**
- `usecols` at read time cuts load memory ~6× → ~50 MB/month.
- 85 months pruned ≈ ~4.25 GB. Still too big to ship to a hosted dashboard.

**After parquet compression:**
- ~4.25 GB of pruned CSV → ~1 GB as parquet (columnar + compressed + typed).
- Free hosting (Render / Fly / Heroku) caps RAM at ~256-512 MB.
- 1 GB parquet expands to ~3-5 GB in memory when Dash loads it → the container gets killed before the app finishes starting.

**So the plan is to pre-aggregate:**
- Grain = week + carrier + origin + dest. City/state/distance are 1:1 with the airport so they ride along.
- Measures = `n_flights`, `n_delayed_*`, `n_cancelled`, `n_diverted`, `sum_arr_delay_min`, plus a sum per delay cause.
- Collapses ~50M rows → ~3M rows → **~35 MB parquet**. Loads instantly on any free tier and all the required breakdowns (cause, carrier, origin/dest airport/city, time range, week/month/quarter/year) are derivable from these counts and sums.

In [19]:
import zipfile
with zipfile.ZipFile(zip_path) as z:
    csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    csv_size_mb = z.getinfo(csv_name).file_size / 1e6

print(f"Shape: {df.shape}  |  Rows: {len(df):,}  |  CSV size: {csv_size_mb:.1f} MB")

Shape: (599013, 120)  |  Rows: 599,013  |  CSV size: 296.9 MB


# Phase 2 — Ingest, Aggregate, Upload

Three steps, all kicked off from this notebook:

- **2a** — for each of 85 months, download the zip and write a pruned parquet (only the ~20 columns we actually need).
- **2b** — read all 85 parquets as one dataframe, do a single groupby, write `flights_agg.parquet`.
- **2c** — upload that one small file to S3 (done via the AWS UI, not in code).

Row-level data stays on my laptop. Only the ~35 MB aggregate goes to S3, and that's all the dashboard ever reads.

### 2a) Download + prune every month into per-month parquet

Idempotent — re-running skips months already written.

In [ ]:
import requests, zipfile, urllib3
import pandas as pd
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

KEEP_COLS = [
    "FlightDate", "DayOfWeek", "DepTimeBlk",
    "Marketing_Airline_Network", "Operating_Airline", "Tail_Number",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "Distance",
    "ArrDel15", "DepDel15", "ArrDelayMinutes",
    "Cancelled", "CancellationCode", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]
KEEP_SET = set(KEEP_COLS)
DELAY_CAUSE_COLS = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

RAW_DIR = Path("raw")
CLEAN_DIR = Path("clean")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

URL_TMPL = ("https://transtats.bts.gov/PREZIP/"
            "On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_{y}_{m}.zip")


def download_zip(url, dest, max_attempts=3):
    # - Retries up to 3 times
    # - Validates the result is a real zip (BTS occasionally returns an HTML error page)
    for attempt in range(1, max_attempts + 1):
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"},
                         stream=True, verify=False, timeout=300)
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        if zipfile.is_zipfile(dest):
            return
        dest.unlink(missing_ok=True)
        print(f"  attempt {attempt}: bad zip, retrying...")
    raise RuntimeError(f"Could not fetch a valid zip from {url} after {max_attempts} attempts")


months = [(y, m) for y in range(2018, 2026) for m in range(1, 13) if (2018, 1) <= (y, m) <= (2025, 1)]

for year, month in months:
    tag = f"{year}_{month:02d}"
    zip_path = RAW_DIR / f"{tag}.zip"
    pq_path = CLEAN_DIR / f"{tag}.parquet"

    if pq_path.exists():
        print(f"[skip] {tag}")
        continue

    if zip_path.exists() and not zipfile.is_zipfile(zip_path):
        print(f"  {tag}: corrupt zip on disk, redownloading")
        zip_path.unlink()

    if not zip_path.exists():
        download_zip(URL_TMPL.format(y=year, m=month), zip_path)

    # Notes on the read:
    # - latin-1 accepts any byte (some months have non-UTF8 city names)
    # - usecols callable strips whitespace to match "Operating_Airline " (trailing space)
    with zipfile.ZipFile(zip_path) as z:
        inner = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(inner) as f:
            df_m = pd.read_csv(
                f,
                usecols=lambda c: c.strip() in KEEP_SET,
                low_memory=False,
                encoding="latin-1",
            )
    df_m.columns = df_m.columns.str.strip()

    df_m["FlightDate"] = pd.to_datetime(df_m["FlightDate"])
    df_m[DELAY_CAUSE_COLS] = df_m[DELAY_CAUSE_COLS].fillna(0).astype("float32")

    df_m.to_parquet(pq_path, index=False)
    print(f"[ok]   {tag}  rows={len(df_m):,}")

## 2b) Aggregate all 85 per-month parquets → one small `flights_agg.parquet`

`pd.read_parquet("clean/")` reads the whole directory as one dataframe (~50M rows, ~4 GB). Then one groupby writes the collapsed result (~3M rows, ~35 MB).

Doing one big groupby across all months rather than per-month-then-concat because a week can straddle a month boundary — per-month aggregates would split those weeks into two rows that'd need re-summing later.

In [ ]:
import pandas as pd
from pathlib import Path

flights = pd.read_parquet("clean/")
print(f"Loaded row-level: {len(flights):,} rows")

flights["week_start"] = flights["FlightDate"].dt.to_period("W-SUN").dt.start_time

# A flight = delayed if EITHER arrival OR departure was >15 min late.
# Matches the prompt's literal wording ("take-off or landing").
flights["delayed_either"] = (
    (flights["ArrDel15"] == 1) | (flights["DepDel15"] == 1)
).astype("float32")

origin_attrs = flights[["Origin", "OriginCityName", "OriginState"]].drop_duplicates("Origin")
dest_attrs   = flights[["Dest",   "DestCityName",   "DestState"]].drop_duplicates("Dest")

agg = (
    flights.groupby(
        ["week_start", "Marketing_Airline_Network", "Origin", "Dest"],
        observed=True,
    ).agg(
        n_flights=("FlightDate", "size"),
        # Three delay flavors so the dashboard can toggle basis
        n_delayed_arr=("ArrDel15", "sum"),
        n_delayed_dep=("DepDel15", "sum"),
        n_delayed_either=("delayed_either", "sum"),
        n_cancelled=("Cancelled", "sum"),
        n_diverted=("Diverted", "sum"),
        sum_arr_delay_min=("ArrDelayMinutes", "sum"),
        # Per-cause delay minutes (BTS attributes these to arrival only)
        sum_carrier_delay=("CarrierDelay", "sum"),
        sum_weather_delay=("WeatherDelay", "sum"),
        sum_nas_delay=("NASDelay", "sum"),
        sum_security_delay=("SecurityDelay", "sum"),
        sum_late_aircraft_delay=("LateAircraftDelay", "sum"),
    )
    .reset_index()
    .rename(columns={"Marketing_Airline_Network": "carrier"})
    .merge(origin_attrs, on="Origin", how="left")
    .merge(dest_attrs,   on="Dest",   how="left")
)

out = Path("flights_agg.parquet")
agg.to_parquet(out, index=False)
print(f"Wrote {out} — {len(agg):,} rows, {out.stat().st_size / 1e6:.1f} MB")
agg.head()

### 2c) Upload `flights_agg.parquet` to S3

Done via AWS S3 UI